In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df.head(5)

,Transaction_ID,Date,Customer_ID,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,TXN-0295,2024-04-21,CUST-480,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
1,TXN-0809,2024-03-09,CUST-373,Perhiasan,18,NaN,961081,11215815,Transfer Bank,Makassar
2,TXN-0364,2024-11-18,CUST-175,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,TXN-0869,2025-09-30,CUST-313,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
4,TXN-0326,2025-03-13,CUST-158,Koin,18,10.93,960581,10499150,Transfer Bank,Surabaya


In [30]:
df = df.dropna()
df.isnull().sum()

Transaction_ID    0
Date              0
Customer_ID       0
Gold_Type         0
Karat             0
Weight_Gram       0
Price_Per_Gram    0
Total_Price       0
Payment_Method    0
Store_Location    0
dtype: int64

In [31]:
df = df.drop_duplicates(keep='last')
df.duplicated().sum()

np.int64(0)

In [32]:
colom_numeric = df.select_dtypes(include=np.number).columns
colom_numeric

Index(['Karat', 'Weight_Gram', 'Price_Per_Gram', 'Total_Price'], dtype='object')

## 1.Principal Component Analysis (PCA)

### 1.PCA Standart

In [56]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import PowerTransformer

df_target = df[colom_numeric].copy()

scaler = PowerTransformer(method='yeo-johnson')
pca = PCA(n_components=0.90) #mencarikan jumlah n_components yang mencakup 90% informasi

df_target[colom_numeric] = scaler.fit_transform(df_target[colom_numeric])
principal_components = pca.fit_transform(df_target[colom_numeric])
pca_df = pd.DataFrame(data=principal_components)

df_target = pd.concat([df_target,pca_df],axis=1).drop(colom_numeric,axis=1)

print(f"Jumlah komponen ideal yang terpilih: {pca.n_components_}")
df_target.head(5)

Jumlah komponen ideal yang terpilih: 2


,0,1
0,2.076464,0.304021
2,2.246987,0.878687
3,2.091219,0.524692
5,-1.772347,0.211523
6,-0.357030,-1.460284


### 2.Incremental PCA

In [61]:
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import PowerTransformer

df_target = df[colom_numeric].copy()

scaler = PowerTransformer(method='yeo-johnson')
pca = IncrementalPCA(n_components=2,batch_size=100) #mengolah per 100 baris data agar tidak berat di RAM

df_target[colom_numeric] = scaler.fit_transform(df_target[colom_numeric])
principal_components = pca.fit_transform(df_target[colom_numeric])
pca_df = pd.DataFrame(data=principal_components)

df_target = pd.concat([df_target,pca_df],axis=1).drop(colom_numeric,axis=1)
df_target

,0,1
0,2.076083,0.305399
2,2.245860,0.878867
3,2.090557,0.525599
5,-1.772352,0.211209
6,-0.355533,-1.459977
...,...,...
844,-0.203684,-1.309717
846,1.047880,-1.815699
864,-0.085425,-1.124866
869,1.798013,-0.279509


### 3.Sparse PCA

In [63]:
from sklearn.decomposition import SparsePCA
from sklearn.preprocessing import PowerTransformer

df_target = df[colom_numeric].copy()

scaler = PowerTransformer(method='yeo-johnson')
pca = SparsePCA(n_components=2,alpha=1,ridge_alpha=0.01,n_jobs=-1) # Membuang fitur tidak penting hingga bernilai 0.

df_target[colom_numeric] = scaler.fit_transform(df_target[colom_numeric])
principal_components = pca.fit_transform(df_target[colom_numeric])
pca_df = pd.DataFrame(data=principal_components)

df_target = pd.concat([df_target,pca_df],axis=1).drop(colom_numeric,axis=1)
df_target

,0,1
0,1.754779,1.015972
2,1.680823,1.608471
3,1.680391,1.225558
5,-1.694140,-0.430485
6,0.270324,-1.479311
...,...,...
844,0.345208,-1.286635
846,1.683515,-1.313970
864,0.375946,-1.073698
869,1.736999,0.374896


## 2.Linear Discriminant Analysis (LDA)

### 1.LDA Sederhana